# Government of Canada Algorithmic Impact Assessments
## JSONL-backed analysis report

This notebook analyzes the same unified bilingual JSON Lines dataset used by the web application. The JSONL combines usable AIA JSON resources published on Open Canada with bilingual JSON reconstructed from official English and French AIA PDFs when no usable JSON resource exists.

If the JSONL file is not present locally, the notebook runs `scripts/build_aia_jsonl.py` to build it directly from the Open Canada catalogue and `recovered_aia_json`. No Google Drive or Colab-specific storage functions are used.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
OUTPUT_DIR = Path(os.environ.get('AIA_REPORT_OUTPUT_DIR', ROOT / 'report-output'))
JSONL_PATH = Path(os.environ.get('AIA_JSONL_PATH', ROOT / 'public/aia-analysis-data/aia-results.jsonl'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not JSONL_PATH.exists():
    subprocess.run([
        sys.executable,
        'scripts/build_aia_jsonl.py',
        '--output', str(JSONL_PATH),
        '--summary-output', str(JSONL_PATH.with_name('aia-results-summary.json')),
        '--recovered-dir', 'recovered_aia_json',
    ], check=True)

records = [
    json.loads(line)
    for line in JSONL_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
]

rows = []
for record in records:
    derived = record.get('derived') or {}
    rows.append({
        'package_id': record.get('package_id'),
        'package_title_en': record.get('title_en'),
        'package_title_fr': record.get('title_fr'),
        'organization_en': record.get('organization_en'),
        'organization_fr': record.get('organization_fr'),
        'publication_date': record.get('metadata_created'),
        'publication_year': str(record.get('metadata_created') or '')[:4],
        'aia_version': record.get('version'),
        'source': record.get('source'),
        'project_phase': derived.get('project_phase'),
        'impact_level': derived.get('impact_level_label'),
        'completeness_pct': derived.get('completeness_pct'),
        'nonconditional_completeness_pct': derived.get('nonconditional_completeness_pct'),
        'dataset_url': record.get('dataset_url'),
        'resource_url': record.get('resource_url'),
    })

df = pd.DataFrame(rows)
df['publication_year'] = df['publication_year'].where(df['publication_year'].str.fullmatch(r'\d{4}', na=False))
display(Markdown(f'**AIA records:** {len(df):,}  \
**Published JSON:** {(df.source == "published").sum():,}  \
**Recovered JSON:** {(df.source == "recovered").sum():,}'))
display(df.head())

## Core analysis

The plots below cover publication volume, institutional coverage, questionnaire versions, project phase, JSON source, structural completeness, and the phase-aware data-quality/public-availability question pairs. Recovered records are analyzed alongside correctly published JSON-backed records.

In [ ]:
def save_current(name: str) -> None:
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / name, dpi=160, bbox_inches='tight')
    plt.show()

year_counts = df.dropna(subset=['publication_year']).groupby('publication_year').size().sort_index()
plt.figure(figsize=(10, 5))
year_counts.plot(kind='bar')
plt.title('Algorithmic Impact Assessments by publication year')
plt.xlabel('Publication year')
plt.ylabel('AIA count')
save_current('01-aia-by-year.png')

org_counts = df['organization_en'].fillna('Unknown').replace('', 'Unknown').value_counts().head(15).sort_values()
plt.figure(figsize=(10, 7))
org_counts.plot(kind='barh')
plt.title('Organizations with the most AIAs')
plt.xlabel('AIA count')
plt.ylabel('Organization')
save_current('02-aia-by-organization.png')

source_counts = df['source'].fillna('unknown').value_counts()
plt.figure(figsize=(7, 5))
source_counts.plot(kind='bar')
plt.title('AIA JSON source')
plt.xlabel('Source')
plt.ylabel('AIA count')
save_current('03-json-source.png')

version_counts = df['aia_version'].fillna('Unknown').value_counts().sort_values()
plt.figure(figsize=(8, 5))
version_counts.plot(kind='barh')
plt.title('AIA questionnaire versions')
plt.xlabel('AIA count')
plt.ylabel('Version')
save_current('04-aia-versions.png')

phase_counts = df['project_phase'].fillna('Unknown').value_counts()
plt.figure(figsize=(7, 5))
phase_counts.plot(kind='bar')
plt.title('AIA project phase')
plt.xlabel('Project phase')
plt.ylabel('AIA count')
save_current('05-project-phase.png')

completeness = (
    df.dropna(subset=['completeness_pct', 'nonconditional_completeness_pct'])
      .groupby('organization_en')[['completeness_pct', 'nonconditional_completeness_pct']]
      .mean()
      .sort_values('completeness_pct', ascending=False)
      .head(15)
      .sort_values('completeness_pct')
)
plt.figure(figsize=(10, 8))
completeness.plot(kind='barh', ax=plt.gca())
plt.title('Average response completeness by organization')
plt.xlabel('Percent complete')
plt.ylabel('Organization')
plt.xlim(0, 100)
save_current('06-completeness-by-organization.png')

## Data quality and public availability

These four paired plots restore the phase-aware data-quality analysis from the earlier report. For each AIA, the Design or Implementation field is selected from its saved project phase. The GBA Plus public-availability field resolves to question **6** for `v0.x` surveys and question **7** for `v1.x` surveys. The charts count only saved **Yes/No** answers; a missing conditional follow-up is not treated as No.

The separate question about posting system data on the Open Government Portal remains excluded.

In [ ]:
DATA_QUALITY_PAIRS = [
    {
        'title': 'Bias testing and public availability',
        'first_label': 'Bias-testing process documented',
        'second_label': 'Process publicly available',
        'first_suffixes': ('1',),
        'second_suffixes': ('2',),
        'filename': '07-data-quality-bias-public.png',
    },
    {
        'title': 'Data-quality resolution and public availability',
        'first_label': 'Resolution process documented',
        'second_label': 'Process publicly available',
        'first_suffixes': ('3',),
        'second_suffixes': ('4',),
        'filename': '08-data-quality-resolution-public.png',
    },
    {
        'title': 'Unreliable-data risk management and public availability',
        'first_label': 'Risk-management process documented',
        'second_label': 'Process publicly available',
        'first_suffixes': ('8',),
        'second_suffixes': ('9',),
        'filename': '09-data-quality-unreliable-public.png',
    },
    {
        'title': 'GBA Plus analysis and public availability',
        'first_label': 'GBA Plus analysis undertaken',
        'second_label': 'Findings publicly available',
        'first_suffixes': ('5',),
        'second_suffixes': None,
        'gba_public': True,
        'filename': '10-data-quality-gba-public.png',
    },
]

def _substantive(value) -> bool:
    if value is None:
        return False
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, (list, tuple, dict, set)):
        return bool(value)
    return True

def _binary_answer(value) -> str | None:
    if not _substantive(value):
        return None
    normalized = str(value).strip().casefold()
    if normalized in {'yes', 'oui'} or normalized.startswith('item1'):
        return 'Yes'
    if normalized in {'no', 'non'} or normalized.startswith('item2'):
        return 'No'
    return None

def _gba_public_suffixes(version: str) -> tuple[str, ...]:
    normalized = str(version or '').strip().casefold()
    normalized = normalized.removeprefix('version').strip().removeprefix('v').removeprefix('.')
    return ('7', '6') if normalized.startswith('1.') else ('6', '7')

def _resolve_data_quality_answer(record: dict, suffixes: tuple[str, ...]):
    data = record.get('data') or {}
    phase = str((record.get('derived') or {}).get('project_phase') or '')
    phases = [phase] if phase in {'Design', 'Implementation'} else []
    phases.extend(candidate for candidate in ('Design', 'Implementation') if candidate not in phases)
    for phase_name in phases:
        for suffix in suffixes:
            field = f'dataQuality{phase_name}{suffix}'
            if _substantive(data.get(field)):
                return data.get(field)
    return None

coverage_rows = []
for pair in DATA_QUALITY_PAIRS:
    counts = {
        pair['first_label']: {'Yes': 0, 'No': 0},
        pair['second_label']: {'Yes': 0, 'No': 0},
    }
    for record in records:
        first = _binary_answer(_resolve_data_quality_answer(record, pair['first_suffixes']))
        second_suffixes = (
            _gba_public_suffixes(record.get('version'))
            if pair.get('gba_public')
            else pair['second_suffixes']
        )
        second = _binary_answer(_resolve_data_quality_answer(record, second_suffixes))
        if first in {'Yes', 'No'}:
            counts[pair['first_label']][first] += 1
        if second in {'Yes', 'No'}:
            counts[pair['second_label']][second] += 1

    pair_counts = pd.DataFrame(counts).T[['Yes', 'No']]
    ax = pair_counts.plot(kind='bar', stacked=False, figsize=(9, 5), width=0.78)
    ax.set_title(pair['title'])
    ax.set_xlabel('Question')
    ax.set_ylabel('AIA count')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Answer', bbox_to_anchor=(1.02, 1), loc='upper left')
    for container in ax.containers:
        ax.bar_label(
            container,
            labels=[str(int(value)) if value > 0 else '' for value in container.datavalues],
            padding=3,
        )
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / pair['filename'], dpi=160, bbox_inches='tight')
    plt.show()

    for question, values in counts.items():
        coverage_rows.append({
            'pair': pair['title'],
            'question': question,
            'yes_count': values['Yes'],
            'no_count': values['No'],
            'answered_count': values['Yes'] + values['No'],
        })

data_quality_coverage = pd.DataFrame(coverage_rows)
display(Markdown('### Data-quality answer coverage'))
display(data_quality_coverage)

## Output tables

Three compact CSVs are written for reuse by downstream analysis. The data-quality plots are generated directly from the unified JSONL so no separate intermediate data file is required.

In [ ]:
assessments_path = OUTPUT_DIR / 'aia_report_assessments.csv'
df.sort_values(['organization_en', 'package_title_en']).to_csv(assessments_path, index=False)

completeness_by_org = (
    df.groupby('organization_en', dropna=False)
      .agg(
          aia_count=('package_id', 'count'),
          average_completeness_pct=('completeness_pct', 'mean'),
          average_nonconditional_completeness_pct=('nonconditional_completeness_pct', 'mean'),
          recovered_count=('source', lambda values: int((values == 'recovered').sum())),
      )
      .reset_index()
      .sort_values(['aia_count', 'organization_en'], ascending=[False, True])
)
completeness_by_org.to_csv(OUTPUT_DIR / 'aia_completeness_by_organization.csv', index=False)

publication_by_year = (
    df.dropna(subset=['publication_year'])
      .groupby(['publication_year', 'source'])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
publication_by_year.to_csv(OUTPUT_DIR / 'aia_publications_by_year.csv', index=False)

display(Markdown(f'Outputs written to `{OUTPUT_DIR}`'))
display(completeness_by_org.head(20))